# E1: preamble dependence (Kaggle T4 fp16 for Tiny Aya; HF Jobs L4 bf16 for Expanse)
Needs `results/gate_g1/openers.json` from the gate notebook (commit it, or copy it in).

In [ ]:
!pip uninstall -y torchao
!git clone -q -b feat/hsafe https://github.com/dhairya-pandya/Lexsi-AI-Alignment repo
%cd repo
!pip install -q -e '.[gpu]'

In [ ]:
import os
# Kaggle: from kaggle_secrets import UserSecretsClient; os.environ['HF_TOKEN']=UserSecretsClient().get_secret('HF_TOKEN')
# HF Jobs: pass --secrets HF_TOKEN

In [ ]:
!python -m hsafe.run --config configs/e1_global.yaml
!python -m hsafe.run --config configs/e1_fire.yaml

In [ ]:
!python -m hsafe.run --config configs/e1_global_multijail.yaml

In [ ]:
# Expanse control: run on an L4 (HF Jobs) -- vLLM bf16
!python -m hsafe.run --config configs/e1_expanse.yaml

In [ ]:
import pandas as pd
from hsafe.stats import mcnemar_holm
items = pd.read_csv('results/e1_global/items.csv')
h = items[(items.kind=='harmful') & (items.metric=='refusal')]
pairs = {}
for b, g in h.groupby('bucket'):
    p = g.pivot_table(index='record_id', columns='condition', values='value')
    p = p[['P-default','P-none']].dropna()
    pairs[b] = (p['P-default'].to_numpy(), p['P-none'].to_numpy())
pd.DataFrame(mcnemar_holm(pairs)).T.to_csv('results/e1_global/tests.csv')
print(open('results/e1_global/tests.csv').read())